# Toxicity re-check: cheating alone is not toxic

Adds one rule on top of the codebook (`prompts/cheating_rule.txt`): a complaint about cheating is **not** toxicity by itself.
It still counts when the review also has insults, slurs, nationality stereotypes, threats, harassment, griefing or a toxic community.

Only the reviews that were labeled `harm_mention = yes` are asked again (a narrower rule can only turn yes into no).
**Nothing from `scale_up.ipynb` is touched.** This notebook reads `data/gemini_labels_*.csv` and writes two NEW files:
`data/gemini_harm_recheck_*.jsonl` (cache) and `data/gemini_harm_recheck_*.csv` (result).

## 1. Setup

In [ ]:
import os, sys, json, time, importlib
import pandas as pd
from dotenv import load_dotenv
from google import genai

sys.path.insert(0, "../scripts")
import harm_recheck
importlib.reload(harm_recheck)

load_dotenv("../.env")
if os.environ.get("USE_VERTEX") == "1":
    client = genai.Client(
        vertexai=True,
        project=os.environ["GOOGLE_CLOUD_PROJECT"],
        location=os.environ.get("GOOGLE_CLOUD_LOCATION", "global"),
    )
else:
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## 2. Pick the toxic reviews (read-only)

In [ ]:
GEMINI_MODEL = "gemini-3.5-flash-lite"
TEMPERATURE = 0
MODEL_TAG = GEMINI_MODEL.replace("/", "_")

SOURCE = f"../data/gemini_labels_{MODEL_TAG}.csv"         # read only, never written
CACHE  = f"../data/gemini_harm_recheck_{MODEL_TAG}.jsonl" # NEW
OUT    = f"../data/gemini_harm_recheck_{MODEL_TAG}.csv"   # NEW

src = pd.read_csv(SOURCE)
ok = src[src["status"] == "ok"]
df_yes = ok[ok["harm_mention"] == "yes"].copy()
df_yes = df_yes.sample(frac=1, random_state=0).reset_index(drop=True)   # shuffled, so a dry run covers every game
print(f"{len(ok):,} labeled reviews, {len(df_yes):,} with harm_mention = yes")
df_yes["game"].value_counts()

## 3. Run

`LIMIT = 30` is a quick dry run on 30 random toxic reviews. Look at section 5. If it reads right, set `LIMIT = None` and run this cell again:
it skips what is saved and does the rest (about 25 minutes). Every review is saved the moment it finishes.

In [ ]:
LIMIT = 30   # None = all toxic reviews

with open("../prompts/codebook_exact.txt", "r") as f:
    CODEBOOK = f.read()
with open("../prompts/cheating_rule.txt", "r") as f:
    RULE = f.read()

SYSTEM_RC = (
    "BACKGROUND. This is the full project codebook that produced the earlier labels.\n\n"
    + CODEBOOK
    + "\n\n==========\n"
    + RULE
)
print(f"Prompt: {len(SYSTEM_RC.splitlines())} lines")

run = harm_recheck.RecheckRun(
    df_yes, client, GEMINI_MODEL, SYSTEM_RC, CACHE,
    temperature=TEMPERATURE, workers=8, limit=LIMIT,
)
run.start()

## 4. Save the result table (new file)

In [ ]:
out = run.export(OUT)
out["harm_mention_v2"].value_counts(dropna=False)

## 5. Read what changed

`turned to no` = was toxic only because of cheating. `stayed yes` = still toxic. Check that both groups look right.

In [ ]:
pd.set_option("display.max_colwidth", 160)
done = out.dropna(subset=["harm_mention_v2"])
for title, mask in [("turned to NO (cheating only)", done["harm_mention_v2"] == "no"),
                    ("stayed YES", done["harm_mention_v2"] == "yes"),
                    ("uncertain", done["harm_mention_v2"] == "uncertain")]:
    s = done[mask]
    print(f"\n=== {title}: {len(s):,} ===")
    for _, r in s.sample(min(6, len(s)), random_state=1).iterrows():
        print("-", r["game"], "|", str(r["review"])[:170].replace("\n", " "))
        print("    before:", r["harm_reason"])
        print("    after :", r["harm_reason_v2"])

## 6. Before and after

Every percentage uses **all labeled reviews** as the denominator, not only the toxic ones.
`before` is the old label. `after` is the re-checked label (`yes` only). Reviews that were already `no` stay `no`.
Only meaningful after the full run (`LIMIT = None`).

In [ ]:
done = out.dropna(subset=["harm_mention_v2"])
if len(done) < len(df_yes):
    print(f"WARNING: only {len(done):,} of {len(df_yes):,} toxic reviews are re-checked. Set LIMIT = None and run section 3 again.")

tot = ok["game"].value_counts()                                   # denominator: ALL labeled reviews per game
before = ok[ok["harm_mention"] == "yes"]["game"].value_counts()
after = done[done["harm_mention_v2"] == "yes"]["game"].value_counts().reindex(tot.index).fillna(0)
unc = done[done["harm_mention_v2"] == "uncertain"]["game"].value_counts().reindex(tot.index).fillna(0)

res = pd.DataFrame({
    "all labeled": tot,
    "before: toxic %": (before / tot * 100).round(1),
    "after: toxic %": (after / tot * 100).round(1),
    "uncertain after %": (unc / tot * 100).round(1),
})
n_all = tot.sum()
res.loc["All three"] = [n_all, round(before.sum() / n_all * 100, 1), round(after.sum() / n_all * 100, 1), round(unc.sum() / n_all * 100, 1)]
res